In [ ]:
import os
import faiss
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from dotenv import load_dotenv

In [5]:
load_dotenv()

True

In [6]:
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

## Setting Up Models

In [30]:
# embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

In [34]:
embeddings.embed_query("What is the capital of France?")

[0.042022705078125,
 0.016387939453125,
 0.0305938720703125,
 0.025177001953125,
 -0.0227203369140625,
 -0.00348663330078125,
 -0.01654052734375,
 0.0146331787109375,
 0.01041412353515625,
 -0.009307861328125,
 0.00516510009765625,
 -0.025421142578125,
 -0.06201171875,
 -0.01326751708984375,
 -0.01551055908203125,
 0.02197265625,
 -0.0056304931640625,
 0.0208282470703125,
 0.0731201171875,
 -0.0245361328125,
 0.00240325927734375,
 -0.01259613037109375,
 -0.038818359375,
 0.0118560791015625,
 0.06280517578125,
 0.004913330078125,
 -0.04541015625,
 -0.00823974609375,
 0.0014629364013671875,
 0.038238525390625,
 0.04217529296875,
 -0.02447509765625,
 -0.003143310546875,
 0.04180908203125,
 -0.0239410400390625,
 -0.040771484375,
 -0.034759521484375,
 -0.040802001953125,
 0.0193634033203125,
 0.0299072265625,
 -0.0023193359375,
 -0.0119171142578125,
 0.00801849365234375,
 0.01279449462890625,
 -0.02740478515625,
 -0.029998779296875,
 -0.00980377197265625,
 -0.038421630859375,
 -0.0328674316

In [9]:
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

In [10]:
llm.invoke("What is the capital of France?").content[0]["text"]

'The capital of France is Paris.'

## Chatting with Model

In [12]:
while True:
    user_input = input("You: ")
    if user_input.lower() in ["exit", "quit"]:
        break
    response = llm.invoke(user_input)
    print("AI:", response.content[0]["text"])

AI: Hello! How can I help you today?
AI: As of **October 2024**, Nvidia has not officially announced or released the RTX 5070, and there is no official pricing information.

However, based on industry leaks, historical pricing patterns, and the typical positioning of "70-series" cards, here is what analysts and hardware experts currently expect:

### 1. Expected Price Range
Most tech analysts and leakers suggest that the RTX 5070 will likely launch in the **$599 to $699 USD** range. 
*   **The RTX 4070** launched at $599.
*   **The RTX 4070 Super** launched at $599.
*   Given Nvidia’s recent trends of increasing prices for performance gains and the expected integration of GDDR7 memory (which is more expensive to produce), it is unlikely to be cheaper than $599. Some analysts speculate it could hit $649 or $699 depending on the amount of VRAM Nvidia decides to include.

### 2. Expected Release Date
Current rumors point to a launch in **early 2025**. Many industry insiders (including pro

## Building Basic RAG Pipeline

In [13]:
file_path = "data/deepseek_v3_technical_report.pdf"

In [14]:
pages = PyPDFLoader(file_path=file_path).load()
len(pages)

53

### Perform Chunking

In [17]:
chunker = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
)

data = chunker.split_documents(pages)

In [18]:
len(data)

502

In [20]:
print(data[0].page_content)

DeepSeek-V3 Technical Report
DeepSeek-AI
research@deepseek.com
Abstract
We present DeepSeek-V3, a strong Mixture-of-Experts (MoE) language model with 671B total
parameters with 37B activated for each token. To achieve efficient inference and cost-effective
training, DeepSeek-V3 adopts Multi-head Latent Attention (MLA) and DeepSeekMoE architec-
tures, which were thoroughly validated in DeepSeek-V2. Furthermore, DeepSeek-V3 pioneers


### Creating Vector Database

In [36]:
index = faiss.IndexFlatL2(1536)

In [37]:
vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [38]:
vector_store.add_documents(data)

['2825df24-0185-409b-8bcb-3c0e407aa03d',
 'f58321d7-5258-4e43-a038-8a8a3662749e',
 'c959a2cc-4e26-4167-8057-283e3a363da3',
 '3561edd8-f73b-4389-8842-e7141531f920',
 '253c773a-abc4-4029-a508-fa0da541c70a',
 '2b61cd4f-3350-4a64-889d-57d2e0206ec5',
 '98505de3-610d-472f-b4d0-0ec601032f43',
 '0fdddc8a-63e6-41ea-a162-de234b36490f',
 '8b70f6cc-1a24-4c60-a4e4-456d4f5ce355',
 '7aeae3f2-ab08-492a-b351-f0b421d6936b',
 '43a54998-b5ff-498c-99bb-716b5fced0ad',
 '5ce288eb-167c-4645-9c02-d65d9a30ffbd',
 '9e091d7b-2bd5-405d-8c27-854ef69602f5',
 'e97b5aca-4989-40cc-8bf7-8ce071fe34ca',
 'b36dda98-f7de-444e-a7c5-01db4896076c',
 'bc993561-057d-4217-b91e-d79e7d2bc036',
 'd51c5e8e-ff12-4bea-84b5-56defbfd74a8',
 '30033a19-500b-4f03-b732-9ca09b59e176',
 '0f861176-d1cd-4bbe-821e-00616e8de781',
 '460ff784-ebea-407e-9fd3-7b727c936840',
 '6d64d52f-6aa4-4da8-b379-640d61dc8f3a',
 'dd9249fb-4fe7-447c-aa73-6d6137cc2721',
 'be847db2-7126-4eea-b61f-d5cafc56bbd8',
 '858ae4c0-1c7c-4de9-a08a-1197ed2f5137',
 '4e448313-5c32-

In [39]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

In [41]:
retrieved_documents = retriever.invoke("What is DeepSeek-V3 model about?")
retrieved_documents

[Document(id='740778e4-689b-4fcc-8a70-fb9618b1de7c', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-02-19T02:11:22+00:00', 'author': '', 'keywords': '', 'moddate': '2025-02-19T02:11:22+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/deepseek_v3_technical_report.pdf', 'total_pages': 53, 'page': 5, 'page_label': '6'}, page_content='other minor details not explicitly mentioned, DeepSeek-V3 adheres to the settings of DeepSeek-\nV2 (DeepSeek-AI, 2024c).\n2.1. Basic Architecture\nThe basic architecture of DeepSeek-V3 is still within the Transformer (Vaswani et al., 2017)\nframework. For efficient inference and economical training, DeepSeek-V3 also adopts MLA\nand DeepSeekMoE, which have been thoroughly validated by DeepSeek-V2. Compared with'),
 Document(id='2825df24-0185-409b-8bcb-3c0e407aa03d', metadata={'prod

### Creating Prompt Template

In [42]:
template = """ Answer the question based only on the following context:
{context}

Question: {question}
"""

In [ ]:
prompt = PromptTemplate.from_template(template)

### Creating Chain

In [45]:
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

In [ ]:
rag_chain = {"context": retriever | format_docs, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()

In [47]:
rag_chain.invoke("What is DeepSeek-V3 model about?")

'Based on the provided text, DeepSeek-V3 is a strong Mixture-of-Experts (MoE) language model with 671B total parameters (with 37B activated for each token). It is built within the Transformer framework and adopts Multi-head Latent Attention (MLA) and DeepSeekMoE architectures to achieve efficient inference and cost-effective training. Additionally, it demonstrates exceptional performance on math benchmarks.'